# TDAH and MI result generation

Notebook runner for training from best Optuna configuration and CSV evaluation.

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

src_path = str(PROJECT_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)


In [ ]:
import pickle

from stf_kernelshap.evaluation.pipeline import (
    evaluate_all_windows_to_single_csv,
    train_tdah_with_best_optuna_config,
)
from stf_kernelshap.data import get_segmented_data


In [ ]:
model_name = "shallowconvnet"  # "eegnet", "shallowconvnet", "tgarnet"
folds_path = "Data/TDAH/folds.pkl"
journal_file = f"Models/TDAH/Optuna/study_{model_name}_TDAH.journal"
study_name = f"study_{model_name}_TDAH"

base_model_args = {
    "nb_classes": 2,
    "Chans": 19,
    "Samples": 512,
}


In [ ]:
with open(folds_path, "rb") as f:
    folds = pickle.load(f)

X, y, sbjs, _ = get_segmented_data(
    path_adhd="Data/TDAH/ieee/ADHD_group",
    path_control="Data/TDAH/ieee/Control_group",
)


In [ ]:
results = train_tdah_with_best_optuna_config(
    model_name=model_name,
    study_name=study_name,
    journal_file=journal_file,
    base_model_args=base_model_args,
    X=X,
    y=y,
    sbjs=sbjs,
    folds=folds,
    epochs=2,
    batch_size=16,
    seed=34,
    seed_gap=100,
    n_repeats=2,
)

df_predictions = results["subject_predictions_df"]
df_predictions


In [ ]:
df_results = evaluate_all_windows_to_single_csv(
    output_csv="Results/MI_results.csv",
)
df_results
